# MeS addition-product inputs

Build four product guesses per selected diene site and the MeS reference. This requires the curated 7_MAAtest_DA.csv and external optimized diene structures. Gaussian optimization is a separate step.

Original notebooks and all saved outputs are preserved in `notebooks/archive/discovery_before_split_20260928/`. This reorganized notebook has not been executed.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "Code" / "model_train.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter within the CycloAddGenerator repository.")
for folder in (REPO_ROOT, REPO_ROOT / "notebooks",
               REPO_ROOT / "notebooks/04_discovery", REPO_ROOT / "notebooks/05_mechanism"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.chdir(REPO_ROOT)


In [ ]:
from _mes_common import *
from _workflow_paths import require_file
display(pd.DataFrame({"path": [str(INPUT_CSV), str(OUTPUT_DIR), str(ALL_SMILES_CSV)],
                      "exists": [INPUT_CSV.is_file(), OUTPUT_DIR.is_dir(), ALL_SMILES_CSV.is_file()]}))


# Methanethiolate (MeS-) attack and C-S transition-state scan

This notebook builds Gaussian inputs for methanethiolate attack at either terminal
atom of a selected 4pi site.

Workflow:

1. Build four MeS- addition-product guesses per row: attack at atom `a` or `b`,
   from the `up` or `down` face.
2. Optimize those product guesses with Gaussian.
3. Read each optimized product and measure its forming `4pi-C--S` bond length
   `d0`.
4. Generate 26 structures with `C--S = factor * d0`, where
   `factor = 1.00, 1.02, ..., 1.50`.
5. Write either independent TS-search inputs (default) or fixed-distance relaxed
   scan inputs for every point, with all `.gjf` files flattened directly into
   `OUTPUT_DIR / "ts_scan"`.

The generated structures are **TS candidates**, not guaranteed transition states.
Confirm a converged TS by one imaginary frequency along C--S bond formation and,
ideally, by IRC calculations.


In [ ]:
require_file(INPUT_CSV)
RUN_PRODUCT_GENERATION = True
RUN_ALL = True
LIMIT = 1

if RUN_PRODUCT_GENERATION:
    product_manifest, product_failed = generate_methanethiolate_product_inputs(
        input_csv=INPUT_CSV,
        output_dir=OUTPUT_DIR,
        limit=None if RUN_ALL else LIMIT,
        overwrite=False,
    )
    display(product_manifest.head())


## Repair completed product calculations

In [ ]:
main.error_improve(str(OUTPUT_DIR), "product")
